# Feature selection
define the feature sets used by every model. Two sets, identical except for `Monthly_Salary`.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.config import DATA_PATH
from src.imbalance import create_imbalance
from src.features import encode_features, select_feature_sets

target = 'Performance_Score'
df = pd.read_csv(DATA_PATH)
df_imb = create_imbalance(df, target)

X = encode_features(df_imb, target)
y = df_imb[target]
X.shape

(64271, 34)

## correlation threshold ?

In [2]:
corr = X.corrwith(y).abs()
noise_level = 2 / np.sqrt(len(X))

print('features              :', len(corr))
print('|corr| < 0.001       :', int((corr < 0.001).sum()))
print(f'|corr| < {noise_level:.4f} (noise) :', int((corr < noise_level).sum()))

features              : 34
|corr| < 0.001       : 5
|corr| < 0.0079 (noise) : 30


- A threshold of 0.001 removes almost nothing, while most correlations below about 0.008 cannot be told apart from zero with this number of rows.
- A linear correlation says nothing about non-linear models: the salary matters through its interaction with the job title.
- Selecting on the target before the split uses the test rows too (a mild leak).

So the selection below does not use the target.

## Rules

- Drop `Hire_Year`: redundant with `Years_At_Company` (correlation -0.99).
- Drop the `Job_Title` dummies: flat on their own, and together with the salary they decode the score (see the ablation at the end).
- Two sets: `salary` and `no_salary`.

In [3]:
sets = select_feature_sets(X)
for name, Xs in sets.items():
    print(name, Xs.shape)
sets['no_salary'].columns.tolist()

salary (64271, 26)
no_salary (64271, 25)


['Age',
 'Years_At_Company',
 'Education_Level',
 'Work_Hours_Per_Week',
 'Projects_Handled',
 'Overtime_Hours',
 'Sick_Days',
 'Remote_Work_Frequency',
 'Team_Size',
 'Training_Hours',
 'Promotions',
 'Employee_Satisfaction_Score',
 'Resigned',
 'Department_Customer Support',
 'Department_Engineering',
 'Department_Finance',
 'Department_HR',
 'Department_IT',
 'Department_Legal',
 'Department_Marketing',
 'Department_Operations',
 'Department_Sales',
 'Gender_Female',
 'Gender_Male',
 'Gender_Other']

In [4]:
assert 'Monthly_Salary' in sets['salary'] and 'Monthly_Salary' not in sets['no_salary']
assert sets['salary'].shape[1] == sets['no_salary'].shape[1] + 1
for Xs in sets.values():
    assert 'Hire_Year' not in Xs and 'Employee_ID' not in Xs
    assert not any(c.startswith('Job_Title_') for c in Xs)
    assert not Xs.isna().any().any()

## Ablation set (used later)

Same selection, but keeping the `Job_Title` dummies. Used in the modeling steps to measure how much the job title adds when the salary is available.

In [5]:
sets_job = select_feature_sets(X, with_job_title=True)
for name, Xs in sets_job.items():
    print(name, Xs.shape)

salary (64271, 33)
no_salary (64271, 32)


## Notes

- **Feature sets**: `salary` has 26 features and `no_salary` 25 (64,271 rows). They differ only by `Monthly_Salary`. From the 34 encoded features, 8 are dropped: `Hire_Year` (redundant with `Years_At_Company`, correlation -0.99) and the 7 `Job_Title` dummies.
- **Why no correlation filter**: with 34 features, only 5 have |corr| < 0.001, so the usual threshold removes almost nothing, whereas 30 of 34 are below the noise level 2/√n ≈ 0.0079. Only `Monthly_Salary` (0.35) is clearly above it. The three others slightly above (Work_Hours_Per_Week, Job_Title_Specialist, Department_Marketing, between 0.008 and 0.011) are expected by chance: testing 34 features at a 5 % level produces about 2 false positives.
- A linear correlation also misses the interaction between salary and job title, and a selection based on the target, done before the split, would use the test rows. The selection therefore depends on no target information.
- **Why Job_Title is excluded from the main sets**: alone it is flat, and together with the salary it decodes the score. Excluding it keeps the comparison with and without salary clean and comparable with the original study, which dropped it through its correlation filter.
- **Ablation set**: `select_feature_sets(X, with_job_title=True)` keeps the dummies (33 / 32 features). It will be used in the modeling steps to measure what the job title adds when the salary is available. A near-perfect score there would confirm the leakage.